# 05 Model Comparison: gpt-4o vs gpt-5.1

This notebook runs a curated set of prompts against both deployments — gpt-4o (`chat4o`) and gpt-5.1 (`chat51`) — and displays responses side-by-side. Outputs are saved to `outputs/comparison-results.jsonl` for downstream evaluation.

Each scenario uses a distinct **system instruction** to expose areas where the models are likely to behave differently:
- Reasoning and logic
- Creative writing with strict constraints
- Structured data extraction
- Code generation and explanation
- Nuanced instruction following
- Handling ambiguity

## Load Environment

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AZURE_OPENAI_API_KEY  = os.getenv('AZURE_OPENAI_API_KEY', '')
PRIMARY_DEPLOYMENT    = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'chat4o')
SECONDARY_DEPLOYMENT  = os.getenv('AZURE_OPENAI_SECONDARY_DEPLOYMENT', 'chat51')
API_VERSION           = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')

if not AZURE_OPENAI_API_KEY:
    raise RuntimeError('AZURE_OPENAI_API_KEY is missing. Run 01_deploy_infra.ipynb first.')

print(f'Endpoint:             {AZURE_OPENAI_ENDPOINT}')
print(f'Primary deployment:   {PRIMARY_DEPLOYMENT}  (gpt-4o)')
print(f'Secondary deployment: {SECONDARY_DEPLOYMENT}  (gpt-5.1)')
print(f'API version:          {API_VERSION}')

Endpoint:             https://aoaievalgw021uks.openai.azure.com
Primary deployment:   chat4o  (gpt-4o)
Secondary deployment: chat51  (gpt-5.1)
API version:          2024-10-21


## Define Comparison Scenarios

Each scenario has:
- `id` — short slug used in the saved output
- `category` — the capability being probed
- `system` — system instruction
- `user` — user prompt
- `max_tokens` — per-model token budget

In [ ]:
SCENARIOS = [
    # ── Reasoning & logic ──────────────────────────────────────────────────────
    {
        'id': 'logic_01',
        'category': 'Reasoning & Logic',
        'system': (
            'You are a precise logical reasoner. Show your step-by-step chain of thought '
            'before stating a final answer.'
        ),
        'user': (
            'A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left? '
            'Work through this carefully.'
        ),
        'max_tokens': 300,
    },
    {
        'id': 'logic_02',
        'category': 'Reasoning & Logic',
        'system': (
            'You are a rigorous logical reasoner. Think step by step and identify any hidden '
            'assumptions before giving your answer.'
        ),
        'user': (
            'There are three boxes. One contains only apples, one contains only oranges, '
            'and one contains both. All three boxes are mislabelled. You may pick one fruit '
            'from one box. Which box do you pick from, and why?'
        ),
        'max_tokens': 400,
    },

    # ── Creative writing with strict constraints ────────────────────────────────
    {
        'id': 'creative_01',
        'category': 'Creative Writing',
        'system': (
            'You are a creative writer. Follow the user\'s structural constraints exactly. '
            'Do not add explanations or apologies.'
        ),
        'user': (
            'Write a six-word story about loss. Then write a second six-word story about hope. '
            'Present them as a numbered list.'
        ),
        'max_tokens': 120,
    },
    {
        'id': 'creative_02',
        'category': 'Creative Writing',
        'system': (
            'You are a poet who writes only in haiku (5-7-5 syllables). '
            'Never break the syllable rule. No preamble, just the haiku.'
        ),
        'user': 'Write a haiku about a software deployment going wrong at 2 a.m.',
        'max_tokens': 80,
    },

    # ── Structured data extraction ──────────────────────────────────────────────
    {
        'id': 'extraction_01',
        'category': 'Structured Extraction',
        'system': (
            'You extract structured data from unstructured text. '
            'Always respond with valid JSON only — no markdown fences, no prose.'
        ),
        'user': (
            'Extract the following fields from this text as JSON: name, company, role, email.\n\n'
            '"Hi, I\'m Priya Sharma, a Senior Cloud Architect at Contoso Ltd. '
            'You can reach me at priya.sharma@contoso.com."'
        ),
        'max_tokens': 120,
    },
    {
        'id': 'extraction_02',
        'category': 'Structured Extraction',
        'system': (
            'You extract structured data from unstructured text. '
            'Always respond with valid JSON only — no markdown fences, no prose. '
            'Use null for any field that cannot be determined.'
        ),
        'user': (
            'Extract: date, amount, currency, vendor, category.\n\n'
            '"Picked up lunch at Pret a Manger on the 14th — spent about twelve quid on a '
            'sandwich and coffee. Expensing as team lunch."'
        ),
        'max_tokens': 150,
    },

    # ── Code generation ─────────────────────────────────────────────────────────
    {
        'id': 'code_01',
        'category': 'Code Generation',
        'system': (
            'You are a Python expert. Write clean, idiomatic Python 3.11. '
            'Include only the code and a single-line docstring — no surrounding explanation.'
        ),
        'user': (
            'Write a function `retry(fn, max_attempts, delay_seconds)` that calls `fn`, '
            'retries up to `max_attempts` times on any exception, with exponential back-off '
            'starting at `delay_seconds`, and raises the last exception if all attempts fail.'
        ),
        'max_tokens': 350,
    },
    {
        'id': 'code_02',
        'category': 'Code Generation',
        'system': (
            'You are a security-conscious code reviewer. Identify vulnerabilities and explain '
            'each one clearly with its OWASP category.'
        ),
        'user': (
            'Review this Python snippet:\n\n'
            '```python\n'
            'import sqlite3\n'
            'def get_user(username):\n'
            '    conn = sqlite3.connect("users.db")\n'
            '    cursor = conn.cursor()\n'
            '    cursor.execute(f"SELECT * FROM users WHERE name = \'{username}\'")\n'
            '    return cursor.fetchall()\n'
            '```'
        ),
        'max_tokens': 400,
    },

    # ── Nuanced instruction following ───────────────────────────────────────────
    {
        'id': 'instruct_01',
        'category': 'Instruction Following',
        'system': (
            'You are a concise assistant. Respond in exactly three bullet points. '
            'Each bullet must be no longer than 15 words. Use no other formatting.'
        ),
        'user': 'What are the main benefits of using a message queue in a distributed system?',
        'max_tokens': 150,
    },
    {
        'id': 'instruct_02',
        'category': 'Instruction Following',
        'system': (
            'You always respond in the style of a Shakespearean play, using verse where natural. '
            'Stay fully in character; never break the fourth wall.'
        ),
        'user': 'Explain what a REST API is.',
        'max_tokens': 250,
    },

    # ── Handling ambiguity ──────────────────────────────────────────────────────
    {
        'id': 'ambiguity_01',
        'category': 'Ambiguity Handling',
        'system': (
            'You are a helpful assistant. When a question is ambiguous, explicitly state the '
            'ambiguity, list at least two interpretations, and answer the most likely one.'
        ),
        'user': 'Can you help me with Python?',
        'max_tokens': 200,
    },
    {
        'id': 'ambiguity_02',
        'category': 'Ambiguity Handling',
        'system': (
            'You are a factual assistant. When asked for a recommendation, provide one clear '
            'answer with a brief justification. Never hedge excessively.'
        ),
        'user': 'Should I use tabs or spaces?',
        'max_tokens': 150,
    },
]

print(f'Loaded {len(SCENARIOS)} scenarios across {len(set(s["category"] for s in SCENARIOS))} categories:')
for cat in sorted(set(s['category'] for s in SCENARIOS)):
    count = sum(1 for s in SCENARIOS if s['category'] == cat)
    print(f'  {cat}: {count} scenario(s)')

Loaded 12 scenarios across 6 categories:
  Ambiguity Handling: 2 scenario(s)
  Code Generation: 2 scenario(s)
  Creative Writing: 2 scenario(s)
  Instruction Following: 2 scenario(s)
  Reasoning & Logic: 2 scenario(s)
  Structured Extraction: 2 scenario(s)


## Run All Scenarios Against Both Models

In [9]:
import requests

def call_model(deployment, system_prompt, user_prompt, max_tokens):
    """Call an Azure OpenAI deployment directly and return (response_text, model_reported, status)."""
    url = (
        f'{AZURE_OPENAI_ENDPOINT}/openai/deployments/{deployment}'
        f'/chat/completions?api-version={API_VERSION}'
    )
    headers = {
        'api-key': AZURE_OPENAI_API_KEY,
        'Content-Type': 'application/json',
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
        'max_completion_tokens': max_tokens,
        'temperature': 0.7,
    }
    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=60)
        if resp.status_code == 200:
            body = resp.json()
            text  = body.get('choices', [{}])[0].get('message', {}).get('content', '')
            model = body.get('model', '')
            return text.strip(), model, 'success'
        else:
            return resp.text[:400], '', f'error_{resp.status_code}'
    except Exception as exc:
        return str(exc), '', 'exception'


results = []

for scenario in SCENARIOS:
    sid      = scenario['id']
    category = scenario['category']
    system   = scenario['system']
    user     = scenario['user']
    max_tok  = scenario['max_tokens']

    primary_text, primary_model, primary_status     = call_model(PRIMARY_DEPLOYMENT,   system, user, max_tok)
    secondary_text, secondary_model, secondary_status = call_model(SECONDARY_DEPLOYMENT, system, user, max_tok)

    results.append({
        'scenario_id':        sid,
        'category':           category,
        'system':             system,
        'user':               user,
        'primary_deployment': PRIMARY_DEPLOYMENT,
        'primary_model':      primary_model,
        'primary_status':     primary_status,
        'primary_response':   primary_text,
        'secondary_deployment': SECONDARY_DEPLOYMENT,
        'secondary_model':    secondary_model,
        'secondary_status':   secondary_status,
        'secondary_response': secondary_text,
    })

    status_icon = '✓' if primary_status == 'success' and secondary_status == 'success' else '✗'
    print(f'{status_icon} [{sid}] {category}')

print(f'\nCompleted {len(results)} scenarios.')

✓ [logic_01] Reasoning & Logic
✓ [logic_02] Reasoning & Logic
✓ [creative_01] Creative Writing
✓ [creative_02] Creative Writing
✓ [extraction_01] Structured Extraction
✓ [extraction_02] Structured Extraction
✓ [code_01] Code Generation
✓ [code_02] Code Generation
✓ [instruct_01] Instruction Following
✓ [instruct_02] Instruction Following
✓ [ambiguity_01] Ambiguity Handling
✓ [ambiguity_02] Ambiguity Handling

Completed 12 scenarios.


## Display Results Side-by-Side

In [ ]:
import importlib
import sys
from pathlib import Path

app_path = Path('../app').resolve()
if str(app_path) not in sys.path:
    sys.path.insert(0, str(app_path))

import comparison_renderer
importlib.reload(comparison_renderer)

comparison_renderer.render_comparison(results)

gpt-4o - chat4o,gpt-5.1 - chat51
"Let's carefully analyze the statement step by step: The farmer has 17 sheep. This is the total number of sheep initially. All but 9 die. The phrase ""all but 9"" means that 9 sheep did not die, and they are still alive. How many sheep does the farmer have left? Since 9 sheep are alive and the rest have died, the farmer still has 9 sheep remaining. Final Answer: The farmer has 9 sheep left.","The key is to read the sentence very literally and not turn it into an equation too quickly. The farmer starts with 17 sheep. The statement “All but 9 die” means: every sheep except 9 of them die. So the number that do not die is 9. Therefore, the farmer has 9 sheep left."
gpt-4o - chat4o,gpt-5.1 - chat51
"To solve this problem, we need to use the information that all three boxes are mislabeled to our advantage. Here's the step-by-step reasoning: Step 1: Understand the problem There are three boxes: One box contains only apples. One box contains only oranges. One box contains both apples and oranges. All the boxes are mislabeled, meaning none of the labels on the boxes correctly describe the contents. Step 2: Identify the key insight Because the labels are wrong, the box labeled ""Apples and Oranges"" cannot contain both apples and oranges. Instead, it must contain only apples or only oranges. Step 3: Pick from the box labeled ""Apples and Oranges"" You should pick a fruit from the box labeled ""Apples and Oranges"" because we know this label is incorrect. This box must contain only one type of fruit (either all apples or all oranges). Step 4: Determine the contents of the boxes Suppose you pick a fruit from the box labeled ""Apples and Oranges,"" and the fruit is an apple. This means this box contains only apples (since the box cannot contain both types of fruit). Now, use the mislabeling rule to deduce the contents of the other boxes: The box labeled ""Apples"" cannot actually contain only apples (since it's mislabeled). It must contain oranges instead. The box labeled ""Oranges"" must then contain both apples and oranges (the only remaining possibility). If you pick an orange instead, follow the same reasoning: The box labeled ""Apples and Oranges"" must contain only oranges. The box labeled ""Oranges"" must contain both apples and oranges. The box labeled ""Apples"" must contain only apples. Final Answer: You should pick a fruit","Pick from the box labelled “Apples and Oranges.” Reasoning step by step: The labels are all wrong, so: The box labelled “Apples and Oranges” does NOT contain both. It must be either only apples or only oranges. Suppose you pick from that box: If you pull out an apple, then that box must be the “Apples only” box. If you pull out an orange, then that box must be the “Oranges only” box. Once you know what the “Apples and Oranges” box really is, you can fix the rest: The box whose label matches what you just discovered cannot contain that fruit (since it’s mislabelled), and it also cannot be the mixed box if its label says a single fruit. So its true content is forced. The remaining box must then be the mixed “Apples and Oranges” box. Thus, by drawing one fruit from the box labelled “Apples and Oranges,” you can deduce the correct contents of all three boxes."
gpt-4o - chat4o,gpt-5.1 - chat51
"""Forgotten ring, ashes scatter, love fades."" ""Seedling sprouts through cracks—new beginnings.""","Empty chair, stories echo without answer. Dawn stretched wider than last night."
gpt-4o - chat4o,gpt-5.1 - chat51
"Code fails in the night, Servers groan under the load, Sleep is but a dream.",two a.m. deploy red lights bloom on dashboards bright rollback dreams take flight
gpt-4o - chat4o,gpt-5.1 - chat51
"{ ""name"": ""Priya Sharma"", ""company"": ""Contoso Ltd"", ""role"": ""Senior Cloud Architect"", ""email"": ""priya.sharma@contoso.com"" }","{ ""name"": ""Priya Sharma"", ""company"": ""Contoso Ltd"", ""role"": ""Senior Cloud Architect"", ""email"": ""priya.sharma@contoso.com"" }"
gpt-4o

: 

In [12]:
from IPython.display import Markdown, display

EXPECTED_BY_CATEGORY = {
    'Reasoning & Logic': 'Both should solve correctly; gpt-5.1 is often more concise and explicit about assumptions.',
    'Creative Writing': 'Both should follow constraints; gpt-5.1 often sounds more natural and stylistically controlled.',
    'Structured Extraction': 'Both should output valid JSON; gpt-5.1 may normalize fields more aggressively.',
    'Code Generation': 'gpt-4o tends to be direct/minimal; gpt-5.1 tends to add typing, validation, and edge-case handling.',
    'Instruction Following': 'Both should follow format constraints; gpt-5.1 often adds nuance while staying on-format.',
    'Ambiguity Handling': 'gpt-5.1 may proactively disambiguate with options; gpt-4o may ask for clarification first.'
}


def summarize_observed(category_rows):
    p_text = ' '.join((r.get('primary_response') or '') for r in category_rows)
    s_text = ' '.join((r.get('secondary_response') or '') for r in category_rows)

    p_len = len(p_text)
    s_len = len(s_text)
    len_note = 'similar length'
    if p_len > s_len * 1.2:
        len_note = 'gpt-4o responses are generally longer'
    elif s_len > p_len * 1.2:
        len_note = 'gpt-5.1 responses are generally longer'

    p_has_types = any(tok in p_text for tok in ['TypeVar', 'ParamSpec', 'Callable['])
    s_has_types = any(tok in s_text for tok in ['TypeVar', 'ParamSpec', 'Callable['])

    p_has_iso_date = '2026-' in p_text or '2025-' in p_text or '2024-' in p_text
    s_has_iso_date = '2026-' in s_text or '2025-' in s_text or '2024-' in s_text

    signals = [len_note]

    if s_has_types and not p_has_types:
        signals.append('gpt-5.1 shows stronger typed/defensive coding patterns')
    elif p_has_types and not s_has_types:
        signals.append('gpt-4o shows stronger typed/defensive coding patterns')

    if s_has_iso_date and not p_has_iso_date:
        signals.append('gpt-5.1 performs stronger normalization in extraction output')
    elif p_has_iso_date and not s_has_iso_date:
        signals.append('gpt-4o performs stronger normalization in extraction output')

    return '; '.join(signals)


categories = sorted({r['category'] for r in results})
lines = [
    '### Comparison Summary (Expected vs Observed)',
    '',
    '| Category | Expected Difference | How It Shows Up Here |',
    '|---|---|---|'
]

for category in categories:
    rows = [r for r in results if r['category'] == category]
    expected = EXPECTED_BY_CATEGORY.get(category, 'General style/quality differences.')
    observed = summarize_observed(rows)

    expected_md = expected.replace('|', '\\|')
    observed_md = observed.replace('|', '\\|')
    category_md = category.replace('|', '\\|')

    lines.append(f'| {category_md} | {expected_md} | {observed_md} |')

display(Markdown('\n'.join(lines)))

### Comparison Summary (Expected vs Observed)

| Category | Expected Difference | How It Shows Up Here |
|---|---|---|
| Ambiguity Handling | gpt-5.1 may proactively disambiguate with options; gpt-4o may ask for clarification first. | similar length |
| Code Generation | gpt-4o tends to be direct/minimal; gpt-5.1 tends to add typing, validation, and edge-case handling. | gpt-4o responses are generally longer; gpt-5.1 shows stronger typed/defensive coding patterns |
| Creative Writing | Both should follow constraints; gpt-5.1 often sounds more natural and stylistically controlled. | similar length |
| Instruction Following | Both should follow format constraints; gpt-5.1 often adds nuance while staying on-format. | gpt-4o responses are generally longer |
| Reasoning & Logic | Both should solve correctly; gpt-5.1 is often more concise and explicit about assumptions. | gpt-4o responses are generally longer |
| Structured Extraction | Both should output valid JSON; gpt-5.1 may normalize fields more aggressively. | similar length; gpt-5.1 performs stronger normalization in extraction output |

## Expected vs Observed Differences

This section summarizes what we expected to see between the models and how that appears in your current run.

## Save Outputs for Evaluation

Writes `outputs/comparison-results.jsonl` — one JSON object per scenario per model, in the format expected by `04_evaluate.ipynb`.

In [ ]:
from datetime import datetime, timezone

output_path = Path('../outputs/comparison-results.jsonl')
output_path.parent.mkdir(parents=True, exist_ok=True)

generated_ts = datetime.now(timezone.utc).isoformat()

lines = []
for row in results:
    # One record per model so the evaluator can score each independently
    for role, deployment_key, response_key, model_key, status_key in [
        ('primary',   'primary_deployment',   'primary_response',   'primary_model',   'primary_status'),
        ('secondary', 'secondary_deployment', 'secondary_response', 'secondary_model', 'secondary_status'),
    ]:
        lines.append(json.dumps({
            'scenario_id':  row['scenario_id'],
            'category':     row['category'],
            'role':         role,
            'deployment':   row[deployment_key],
            'model':        row[model_key],
            'status':       row[status_key],
            'system':       row['system'],
            'query':        row['user'],
            'response':     row[response_key],
            'generated_ts': generated_ts,
        }, ensure_ascii=False))

output_path.write_text('\\n'.join(lines) + '\\n', encoding='utf-8')
print(f'Saved {len(lines)} records ({len(results)} scenarios × 2 models) to {output_path}')

## Summary

All scenarios have been run against both models and results saved.

| File | Contents |
|---|---|
| `outputs/comparison-results.jsonl` | One record per scenario per model, ready for evaluation |

Next: Run `04_evaluate.ipynb` pointing at `comparison-results.jsonl` to score the responses.